# profile

> What abhyasa knows about you: your background, goals, interests, the examples that help you, your pace, and what confuses you.

The tutor asks better questions when it knows the learner. A coder gets code examples. A Sanskrit reader gets a Sanskrit example.
This module keeps one `profile` document on the learning shelf. The profile gets data in three ways: a short first conversation, edits by the
learner, and changes that the tutor makes from evidence. The module keeps each change with who made it and why. So the learner can see
each change and undo it. The profile stays in the vault on this machine.

In [ ]:
#| default_exp profile

In [ ]:
#| hide
from fastcore.test import *

Import the modules that this notebook uses.

In [ ]:
#| export
import json, time
from fastcore.utils import *
from abhyasa.store import Learning, _src
from abhyasa.llm import ask_json
from abhyasa.model import HIGH
from abhyasa.method import methods_text, EXPLORE

## The fields

A field is a list of short items, or one value. `PACES` names the three paces. `EXAMPLES` names the kinds of example a learner can like.

In [ ]:
#| export
PROFILE = dict(background=[], goals=[], interests=[], examples=[], pace='steady', session_len=12, confuses=[], notes='', explore='sometimes')
LIST_FIELDS = ('background', 'goals', 'interests', 'examples', 'confuses')
EXAMPLES = ('code', 'maths', 'analogies', 'stories', 'diagrams', 'real cases')
PACES = dict(gentle='small steps: at most one depth above what you know', steady='the usual steps', fast='bigger steps, further past what you know')
LABELS = dict(background='Background', goals='Goals', interests='Interests', examples='Examples that help', pace='Pace',
              session_len='Questions in a session', confuses='What confuses you', notes='Notes', explore='How often to try a new way')
_SRC = _src('profile', 'me')

`profile_text` gives the profile as lines for a prompt. It is empty when the profile has no data. So the prompt for a new learner does not change.

In [ ]:
#| export
def profile_text(p) -> str:
    "The profile as prompt lines; empty when nothing is known."
    out = [f"{LABELS[k]}: {'; '.join(p[k])}" for k in LIST_FIELDS if p.get(k)]
    if p.get('pace', 'steady') != 'steady': out.append(f"Pace: {p['pace']} ({PACES[p['pace']]})")
    if p.get('notes'): out.append(f"Notes: {p['notes']}")
    return '\n'.join(out)

`clean` makes a value fit its field. A list field takes a list or lines of text. The pace must be a known pace. The session length stays between 3 and 40.

In [ ]:
#| export
def clean(k, v):
    "The value `v` made to fit field `k`."
    if k in LIST_FIELDS:
        if isinstance(v, str): v = v.splitlines()
        return list(dict.fromkeys(x.strip(' -*\t') for x in v or () if x and x.strip(' -*\t')))
    if k == 'session_len':
        try: return max(3, min(40, int(v)))
        except (TypeError, ValueError): return PROFILE['session_len']
    if k == 'pace': return v if v in PACES else 'steady'
    if k == 'explore': return v if v in EXPLORE else 'sometimes'
    return str(v or '').strip()

## The document

`profile` reads the document. It uses the defaults for the fields that are not set. `known` is False until the profile has a saved value.

In [ ]:
#| export
@patch
def profile(self:Learning) -> AttrDict:
    "The learner profile, with its list of changes, newest first."
    r = first(self.v.t.docs(where='source=?', where_args=[_SRC]))
    m = json.loads(r['meta'] or '{}') if r else {}
    return AttrDict({**PROFILE, 'changes': [], **{k: v for k, v in m.items() if k in (*PROFILE, 'changes', 'at')}, 'known': bool(r)})

`set_profile` changes fields. It keeps each change with the old value, the new value, who made it (`you`, `onboarding` or `tutor`) and why.

In [ ]:
#| export
@patch
def set_profile(self:Learning, by:str='you', why:str='', **kv) -> list:
    "Change fields of the profile and keep each change with who made it and why. Return the changes."
    p, ch = self.profile(), []
    for k, v in kv.items():
        if k not in PROFILE: continue
        v = clean(k, v)
        if v != p[k]: ch.append(dict(at=time.time(), field=k, old=p[k], new=v, by=by, why=why)); p[k] = v
    if not ch: return []
    m = {**{k: p[k] for k in PROFILE}, 'changes': [*ch[::-1], *p.changes][:200], 'at': time.time()}
    r = self.v.add(profile_text(p) or 'An empty profile.', title='About you', source=_SRC, kind='profile', meta=m)
    if not r.get('changed'): self.v.set_meta(r['doc_id'], **m)
    return ch

`undo_profile` puts back the old value of one change. It finds the change by its time. The undo is a change too, so the list shows all the changes.

In [ ]:
#| export
@patch
def undo_profile(self:Learning, at:float) -> list:
    "Put back the old value of the change made at `at`."
    c = next((c for c in self.profile().changes if abs(c['at'] - float(at)) < 1e-6), None)
    return self.set_profile(by='you', why='you undid a change', **{c['field']: c['old']}) if c else []

@patch
def clear_profile(self:Learning):
    "Remove the profile and its changes."
    if (r := first(self.v.t.docs(where='source=?', where_args=[_SRC]))): self.v.forget(r['id'])

## The first conversation

The learner answers four questions, by voice or by text. One model call turns the answers into fields.

In [ ]:
#| export
ONBOARD_QS = (('who', 'What do you do, and what do you know well already?', 'For example: I write Python each day. I know Sanskrit grammar.'),
              ('want', 'What do you want to learn, and why?', 'For example: how LLMs work, so I can read the papers.'),
              ('how', 'What kind of explanation works for you?', 'Code, maths, analogies, stories, pictures, real cases.'),
              ('time', 'How long do you like to study at one time? Small steps or big ones?', 'For example: twenty minutes, small steps.'))

ONBOARD_SP = """You make a short learner profile from what a learner says about themselves.
Use only what they say; leave a field empty when they say nothing about it. Keep each item under 12 words, in their words where you can."""

def onboard_prompt(answers:dict) -> str:
    "The prompt that turns the onboarding answers into profile fields."
    qa = '\n'.join(f"Q: {q}\nA: {answers.get(k) or '(no answer)'}" for k, q, _ in ONBOARD_QS)
    return (f"{qa}\n\nReturn JSON: {{\"background\": [str], \"goals\": [str], \"interests\": [str], "
            f"\"examples\": [str, each one of {list(EXAMPLES)}], \"pace\": \"gentle\" | \"steady\" | \"fast\", "
            '"session_len": int (questions in one sitting; one question takes about two minutes), "notes": str}')

def onboard(lr:Learning, ask, answers:dict) -> list:
    "Fill the profile from the onboarding answers. Return the changes."
    r = ask_json(ask, onboard_prompt(answers), ONBOARD_SP)
    return lr.set_profile(by='onboarding', why='from what you said when you started', **{k: v for k, v in r.items() if k in PROFILE})

## The tutor learns from evidence

`observe` reads the recent answers: right or not, confidence, hints, misconceptions and think-aloud. A small model call proposes at
most three changes, each with its reason. `observe` applies them immediately and keeps each one with its reason. So the learner sees them and can undo them.

In [ ]:
#| export
OBSERVE_SP = """You keep a short profile of one learner. You change it only from evidence in their recent answers.
Propose at most 3 changes, and none when the evidence is weak. Good changes: a topic that confuses them again and again,
a kind of example that works (they explain well with code or analogies), a pace that fits (many hints and misses: gentle;
quick solid passes: fast), background that their answers show. Each `why` names the evidence in one short sentence, to the
learner as "you", in ASD-STE100 Simplified Technical English."""

def evidence(lr:Learning, n:int=12) -> str:
    "The last `n` final answers as short lines for a prompt: the outcome, the method, the first error, gaps you named, lessons you asked for, and what you said."
    def line(a):
        c = lr.concept(a.concept)
        th = a.get('thread') or []
        first_err = first(t.get('first_error') for t in th if t.get('kind') == 'nudge' and t.get('first_error'))
        said = [t['text'][:160] for t in th if t.get('who') == 'you' and t.get('text')]
        bits = [f"{c.name if c else a.concept} at depth {a.depth}: {a.get('outcome') or ('right' if a.correct else 'wrong')}",
                a.get('method') and f"taught with {a.method}", f"confidence {a.confidence}", f"{a.hints} hints" if a.hints else '',
                first_err and f"first error: {first_err}", a.get('gaps') and f"said they did not know: {', '.join(a.gaps)}",
                a.get('taught') and f"asked for a lesson after it ({a.get('teach_method') or 'same way'})",
                a.misconception and f"misconception: {a.misconception}",
                f"what they said: {' / '.join(said)}" if said else a.answer and f"answer: {a.answer[:160]}", a.thinking and f"think-aloud: {a.thinking[:160]}"]
        return '- ' + '; '.join(b for b in bits if b)
    return '\n'.join([line(a) for a in lr.attempts(limit=n * 3) if a.get('final') and a.get('kind') != 'demo'][:n])

def observe_prompt(p, ev:str, st=None) -> str:
    "The prompt that proposes changes to profile `p` from the evidence `ev` and the method counts in the stats `st`."
    return (f"The profile now:\n{profile_text(p) or '(empty)'}\n\nTheir recent answers, newest first:\n{ev}\n\n"
            + (f"{mt}\n\n" if (mt := methods_text(st or {})) else '')
            + 'Return JSON: {"changes": [{"field": "background" | "interests" | "examples" | "pace" | "confuses", '
            '"add": str or null, "remove": str or null, "value": str or null (for pace), "why": str}]}')

`observe` applies the changes. A list field takes `add` and `remove`. The pace takes `value`. The goals and the notes are for the learner only, so `observe` drops a change to them.

In [ ]:
#| export
def observe(lr:Learning, ask, n:int=12) -> list:
    "Change the profile from the recent answers. Return the changes, each with its reason."
    if not (ev := evidence(lr, n)): return []
    lr.set_stats(calib=dict(calibration(lr)))   # cached here, every few closes, so a question does not read all attempts
    r = ask_json(ask, observe_prompt(lr.profile(), ev, lr.stats()), OBSERVE_SP)
    out = []
    for c in (r.get('changes') or [])[:3] if isinstance(r, dict) else []:
        f, p = c.get('field'), lr.profile()
        if f in ('background', 'interests', 'examples', 'confuses'): new = [x for x in p[f] if x != c.get('remove')] + ([c['add']] if c.get('add') else [])
        elif f == 'pace' and c.get('value'): new = c['value']
        else: continue
        out += lr.set_profile(by='tutor', why=c.get('why', ''), **{f: new})
    return out

## How well you know what you know

A good learner knows what they know. When you feel sure and you are often wrong, you stop checking your own thinking. When you feel unsure and you are often right, you hold back ideas that work. The "Sure?" choice on each answer gives the data. `calibration` counts the first try of each question. It counts how many times you felt sure (60% or more), and how many of those answers were right. It counts the same for unsure. An answer with no confidence, a "don't know" and a demo do not count.

In [ ]:
#| export
def calibration(lr:Learning, n:int=300) -> AttrDict:
    "How often the first try of a question was right, when you felt sure and when you felt unsure."
    seen, r = set(), AttrDict(sure=0, sure_right=0, unsure=0, unsure_right=0)
    for a in reversed(lr.attempts(limit=n)):   # oldest first, so the first try of each question counts
        k = (a.get('session'), a.question)
        if k in seen: continue
        seen.add(k)
        if a.get('kind') == 'demo' or a.get('dont_know') or a.get('confidence') is None: continue
        b = 'sure' if a.confidence >= HIGH else 'unsure'
        r[b], r[f'{b}_right'] = r[b] + 1, r[f'{b}_right'] + bool(a.correct)
    return r

`calibration_text` gives the calibration in plain words, with one line about what it means. With fewer than five answers, it gives an empty text.

In [ ]:
#| export
def calibration_text(r) -> str:
    "The calibration `r` in plain words; empty with fewer than five answers."
    if r.sure + r.unsure < 5: return ''
    out = [f'When you felt sure, you were right in {r.sure_right} of {r.sure}.' if r.sure else '',
           f'When you felt unsure, you were right in {r.unsure_right} of {r.unsure}.' if r.unsure else '']
    if r.sure >= 3 and r.sure_right / r.sure < .7: out.append('You are often sure and wrong. Before you answer, ask: how could this be wrong?')
    elif r.unsure >= 3 and r.unsure_right / r.unsure > .7: out.append('You know more than you think. Trust your first idea a little more.')
    elif r.sure >= 3: out.append('When you feel sure, you are usually right. Your sense of what you know is good.')
    return ' '.join(o for o in out if o)

## Tests

A new shelf has an empty profile, and its prompt text is empty.

In [ ]:
from abhyasa.store import temp_learning
from abhyasa.llm import Fake
lr = temp_learning()
p = lr.profile()
test_eq((p.known, p.pace, p.background, profile_text(p)), (False, 'steady', [], ''))

`set_profile` keeps each change, with who made it. Lines of text become a list.

In [ ]:
ch = lr.set_profile(background='codes in Python\nknows Sanskrit grammar', pace='gentle')
test_eq([c['field'] for c in ch], ['background', 'pace'])
p = lr.profile()
test_eq((p.known, p.background, p.changes[0]['by']), (True, ['codes in Python', 'knows Sanskrit grammar'], 'you'))
assert 'Background: codes in Python; knows Sanskrit grammar' in profile_text(p) and 'Pace: gentle' in profile_text(p)

When a value stays the same, `set_profile` makes no change. A bad pace becomes `steady`, and the session length stays in its range.

In [ ]:
test_eq(lr.set_profile(pace='gentle'), [])
test_eq((clean('pace', 'warp'), clean('session_len', '200'), clean('session_len', 'x')), ('steady', 40, 12))

`undo_profile` puts back the old value, and keeps the undo as a change.

In [ ]:
at = [c for c in lr.profile().changes if c['field'] == 'pace'][0]['at']
lr.undo_profile(at)
test_eq((lr.profile().pace, lr.profile().changes[0]['why']), ('steady', 'you undid a change'))

`onboard` fills fields from the answers with one model call.

In [ ]:
fake = Fake({'make a short learner profile': dict(goals=['understand LLM papers'], examples=['code', 'maths'], session_len=8, pace='steady', notes='')})
ch = onboard(lr, fake, dict(want='LLMs, to read papers', how='code please'))
p = lr.profile()
test_eq((p.goals, p.examples, p.session_len), (['understand LLM papers'], ['code', 'maths'], 8))
test_eq({c['by'] for c in ch}, {'onboarding'})

`observe` changes the profile from attempts and keeps the reason. It drops a change to a field that it must not change.

In [ ]:
lr.put('Attention', 'Weights from query-key dot products.', 'ml')
lr.record('attention', 'What shape is QK^T?', answer='d by d', correct=False, confidence=80, depth=3, misconception='mixes up the axes', final=True)
fake.rules['You keep a short profile'] = dict(changes=[dict(field='confuses', add='tensor shapes in attention', why='You missed a shape question with high confidence.'),
                                                     dict(field='goals', add='hack the planet', why='no')])
ch = observe(lr, fake)
test_eq([c['field'] for c in ch], ['confuses'])
test_eq((lr.profile().confuses, lr.profile().changes[0]['by']), (['tensor shapes in attention'], 'tutor'))
assert 'misconception: mixes up the axes' in fake.calls[-1][0]

With no answers there is no evidence, and no model call.

In [ ]:
n = len(fake.calls)
test_eq(observe(temp_learning(), fake), [])
test_eq(len(fake.calls), n)

`calibration` counts the first try of each question. A pass after a wrong try counts as wrong. A "don't know" and an answer with no confidence do not count.

In [ ]:
kl = temp_learning(); kl.put('Attention', 'Weights from query-key dot products.', 'ml')
def rec(q, ok, conf, **kw): kl.record('attention', q, correct=ok, confidence=conf, session='s1', **kw)
rec('q1', False, 90, final=False); rec('q1', True, 90, final=True)   # sure, but wrong on the first try
rec('q2', True, 90); rec('q3', True, 30); rec('q4', False, 10)
rec('q5', False, None, dont_know=True); rec('q6', True, None)
test_eq(calibration(kl), dict(sure=2, sure_right=1, unsure=2, unsure_right=1))
test_eq(calibration_text(calibration(kl)), '')                       # four answers: too few to say

With enough answers, the text gives the counts and one line about what they mean.

In [ ]:
test_eq(calibration_text(AttrDict(sure=4, sure_right=1, unsure=1, unsure_right=0)),
        'When you felt sure, you were right in 1 of 4. When you felt unsure, you were right in 0 of 1. '
        'You are often sure and wrong. Before you answer, ask: how could this be wrong?')
assert calibration_text(AttrDict(sure=1, sure_right=1, unsure=4, unsure_right=4)).endswith('Trust your first idea a little more.')

`evidence` reads the full conversation of a question. It gets the first error from a nudge, and the ideas that you said you did not know. It also gets a lesson that you asked for, the method, and each thing you said.

In [ ]:
el = temp_learning(); el.put('Neuron', 'z = w . x + b.', 'ml')
th = [dict(who='you', text="I don't know what the loop means"), dict(who='tutor', kind='nudge', text='A loop adds products.', first_error='You use squares.'),
      dict(who='you', text='2(w1x1 + w2x2 + b)')]
el.record('neuron', 'Why does z double?', answer='2z', final=True, outcome='fragile', method='maths', thread=th, gaps=['weighted sum as a loop'], taught=1, teach_method='code')
ev = evidence(el)
for s in ('taught with maths', 'first error: You use squares.', 'said they did not know: weighted sum as a loop', 'asked for a lesson after it (code)', "I don't know what the loop means / 2(w1x1"):
    assert s in ev, s

`observe` keeps the calibration in the stats document. So each new question can read it, and does not have to read all attempts.

In [ ]:
observe(el, Fake({'keep a short profile': dict(changes=[])}))
test_eq(set(el.stats().calib), {'sure', 'sure_right', 'unsure', 'unsure_right'})
test_eq((clean('explore', 'often'), clean('explore', 'always')), ('often', 'sometimes'))

`clear_profile` removes the full profile and its list of changes.

In [ ]:
lr.clear_profile()
test_eq((lr.profile().known, lr.profile().changes), (False, []))

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()